# Full KGE distributions for all interpolation experiments

This notebook reproduces the leave-one-station-out validation and presents full station-level KGE distributions for all four experiments: Raw WRF simulations, Stage 1, Stage 1 + 2, and Ordinary Kriging (OK). Each calendar month is represented by four adjacent boxplots, with 14 station-level KGE values in every box.

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from pykrige.ok import OrdinaryKriging

from utils import gp_interpolator, kge

mpl.rcParams['figure.dpi'] = 150
mpl.rcParams['font.family'] = 'Myriad Pro'

N_MONTHS = 12
MONTH_NAMES = np.array(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                        'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
FIGURE_DIR = Path('figures')
FIGURE_DIR.mkdir(exist_ok=True)

## Load observations, collocated WRF simulations, and station locations

In [2]:
rain_obs = np.loadtxt('data/sta_monthly.csv')
rain_sim_flatten = np.loadtxt('data/wrf_monthly.csv')

sim_sel = np.loadtxt('data/wrf_loc.csv')
sim_idx = sim_sel[:, :2].astype(int)
station_flat_idx = np.ravel_multi_index(
    (sim_idx[:, 0], sim_idx[:, 1]), (120, 160)
)
wrf_sta = rain_sim_flatten[:, station_flat_idx]
sta_loc = np.genfromtxt('data/sta_lookup_new.csv', delimiter=',')[:, 2:]

N_STATIONS = rain_obs.shape[1]
N_YEARS = rain_obs.shape[0] // N_MONTHS

assert rain_obs.shape == wrf_sta.shape
assert rain_obs.shape == (N_YEARS * N_MONTHS, N_STATIONS)
assert sta_loc.shape == (N_STATIONS, 2)

print(f'Loaded {N_YEARS} years and {N_STATIONS} stations.')

Loaded 40 years and 14 stations.


## Reproduce the leave-one-station-out KGE experiment

For each calendar month, one station is withheld. Stage 1 conditions the WRF-derived Gaussian prior on the remaining stations. Stage 1 + 2 then adds an ordinary-kriged residual correction. Negative predictions are truncated to zero, matching the existing validation notebook.

In [3]:
def run_kge_validation(rain_obs, wrf_sta, sta_loc):
    kge_raw = np.zeros((N_MONTHS, N_STATIONS))
    kge_stage1 = np.zeros((N_MONTHS, N_STATIONS))
    kge_stage2 = np.zeros((N_MONTHS, N_STATIONS))

    for month_idx in range(N_MONTHS):
        obs_month = rain_obs[month_idx::N_MONTHS, :]
        sim_month = wrf_sta[month_idx::N_MONTHS, :]
        start = time.time()

        for held_out in range(N_STATIONS):
            station_mask = np.ones(N_STATIONS, dtype=bool)
            station_mask[held_out] = False

            train_obs = obs_month[:, station_mask]
            train_sim = sim_month[:, station_mask]
            train_loc = sta_loc[station_mask, :]
            target_obs = obs_month[:, held_out]
            target_sim = sim_month[:, held_out][:, None]

            gp = gp_interpolator(P=N_STATIONS - 1)
            gp.read_rainfall(obs=train_obs, sim=train_sim)
            with redirect_stdout(StringIO()):
                gp.sn_converge()

            stage1_pred, _ = gp.predict(target_sim)
            stage1_train, _ = gp.predict(train_sim)
            residuals = train_obs - stage1_train.T

            stage2_pred = np.zeros(N_YEARS)
            target_lon = np.array([sta_loc[held_out, 0]])
            target_lat = np.array([sta_loc[held_out, 1]])

            for year_idx in range(N_YEARS):
                ok = OrdinaryKriging(
                    train_loc[:, 0],
                    train_loc[:, 1],
                    residuals[year_idx, :],
                    variogram_model='gaussian',
                    verbose=False,
                    enable_plotting=False,
                )
                kriged_residual, _ = ok.execute('points', target_lon, target_lat)
                stage2_pred[year_idx] = (
                    stage1_pred[year_idx] + kriged_residual.data[0]
                )

            stage1_pred = np.maximum(np.asarray(stage1_pred).squeeze(), 0)
            stage2_pred = np.maximum(stage2_pred, 0)

            kge_raw[month_idx, held_out] = kge(
                target_obs, target_sim.squeeze()
            )
            kge_stage1[month_idx, held_out] = kge(target_obs, stage1_pred)
            kge_stage2[month_idx, held_out] = kge(target_obs, stage2_pred)

        print(f'{MONTH_NAMES[month_idx]}: {time.time() - start:.2f} s')

    return {
        'Raw': kge_raw,
        'Stage 1': kge_stage1,
        'Stage 1 + 2': kge_stage2,
    }

In [4]:
kge_results = run_kge_validation(rain_obs, wrf_sta, sta_loc)
kge_results['Ordinary Kriging'] = np.loadtxt('kriging_kge.csv')

for method, values in kge_results.items():
    assert values.shape == (N_MONTHS, N_STATIONS), (method, values.shape)

print('KGE arrays prepared for:', ', '.join(kge_results))

Jan: 4.17 s
Feb: 4.30 s
Mar: 3.49 s
Apr: 5.86 s
May: 6.20 s
Jun: 6.12 s
Jul: 6.66 s
Aug: 5.22 s
Sep: 5.94 s
Oct: 5.05 s
Nov: 4.55 s
Dec: 4.15 s
KGE arrays prepared for: Raw, Stage 1, Stage 1 + 2, Ordinary Kriging


## Station-level distribution summary

In [5]:
summary_records = []
for method, values in kge_results.items():
    for month_idx, month_name in enumerate(MONTH_NAMES):
        month_values = values[month_idx, :]
        summary_records.append({
            'method': method,
            'month': month_name,
            'minimum': np.min(month_values),
            'q25': np.quantile(month_values, 0.25),
            'median': np.median(month_values),
            'q75': np.quantile(month_values, 0.75),
            'maximum': np.max(month_values),
        })

distribution_summary = pd.DataFrame.from_records(summary_records)
distribution_summary.round(3)

,method,month,minimum,q25,median,q75,maximum
0,Raw,Jan,-0.410,-0.269,0.051,0.125,0.340
1,Raw,Feb,-0.304,-0.012,0.159,0.304,0.474
2,Raw,Mar,-0.331,0.184,0.238,0.294,0.464
3,Raw,Apr,-0.066,0.056,0.164,0.224,0.336
4,Raw,May,-0.417,-0.200,-0.149,-0.062,0.113
5,Raw,Jun,-0.228,-0.089,-0.022,0.078,0.133
6,Raw,Jul,-0.303,-0.199,-0.169,-0.076,0.181
7,Raw,Aug,-0.131,-0.009,0.108,0.182,0.388
8,Raw,Sep,-0.081,0.054,0.129,0.155,0.357
9,Raw,Oct,0.094,0.297,0.346,0.364,0.411


## Four boxplots per calendar month

Boxes show the interquartile range, black lines show medians, whiskers extend to 1.5 times the interquartile range, and open circles show station-level outliers. The vertical scale is set from the complete finite KGE range so no method's outliers are clipped.

In [6]:
method_specs = [
    ('Raw', '#737373'),
    ('Stage 1', '#C47900'),
    ('Stage 1 + 2', '#70A0CD'),
    ('Ordinary Kriging', '#59A14F'),
]
month_positions = np.arange(N_MONTHS, dtype=float)
offsets = np.array([-0.30, -0.10, 0.10, 0.30])
box_width = 0.16

fig, ax = plt.subplots(figsize=(10.5, 5.2))

for method_idx, (method, color) in enumerate(method_specs):
    monthly_values = [
        kge_results[method][month_idx, np.isfinite(kge_results[method][month_idx])]
        for month_idx in range(N_MONTHS)
    ]
    boxplot = ax.boxplot(
        monthly_values,
        positions=month_positions + offsets[method_idx],
        widths=box_width,
        patch_artist=True,
        manage_ticks=False,
        whis=1.5,
        showfliers=True,
        boxprops={'facecolor': color, 'edgecolor': 'black', 'linewidth': 0.9},
        medianprops={'color': 'black', 'linewidth': 1.4},
        whiskerprops={'color': 'black', 'linewidth': 0.9},
        capprops={'color': 'black', 'linewidth': 0.9},
        flierprops={
            'marker': 'o', 'markerfacecolor': 'none',
            'markeredgecolor': color, 'markersize': 3.5, 'linestyle': 'none'
        },
    )

all_finite_kge = np.concatenate([
    values[np.isfinite(values)].ravel() for values in kge_results.values()
])
y_min = np.floor((np.min(all_finite_kge) - 0.05) * 10) / 10
ax.set_ylim(-1.0, 1.0)
ax.axhline(-0.41, color='black', linestyle='--', linewidth=0.9, alpha=0.8)

ax.set_xlim(-0.55, N_MONTHS - 0.45)
ax.set_xticks(month_positions)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('KGE')
ax.yaxis.set_major_locator(mticker.MultipleLocator(0.5))
ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.25))
ax.grid(which='major', axis='y', linestyle='--', alpha=0.45)
ax.grid(which='minor', axis='y', linestyle=':', alpha=0.25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend_handles = [
    Patch(facecolor=color, edgecolor='black', label=method)
    for method, color in method_specs
]
ax.legend(
    handles=legend_handles, loc='upper center', ncol=4, frameon=False,
    bbox_to_anchor=(0.5, 1.13)
)

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / 'kge_boxplot_full_distributions.png',
    dpi=600, bbox_inches='tight'
)
plt.show()

/var/folders/wy/_8r9h_1562q_4cj3h4l5ht0m0000gn/T/ipykernel_11592/2113093170.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
